#  Phil Job Applier Agent - Rachel Briskman & Sanila Chowdhury (Daedalus Devs)

## Team Report #1 (03/24) - Created a function that takes in user's resume and parse it to output the extracted info & uses Skyvern to see how the library works for filling in the fields of a job website

<img src="AI_AGENT_WORKFLOW.png" width="700">

### Installs and Imports

In [ ]:

%pip install chromadb
# %pip install "skyvern==1.0.24"
%pip install skyvern
# %playwright install --with-deps chromium
# %playwright install chromium
%pip install langchain langchain-huggingface
%pip install python-dotenv

print ("Done installing")

In [ ]:
#installing dependencies for browser-use (an alternative to skyvern)
%pip install browser-use playwright
%pip install -U langchain-google-genai
%pip install -U browser-use
# %playwright install chromium

### Import API keys from .env file
#### (Note: This is for local use. Colab imports keys differently)

In [ ]:
import os
from dotenv import load_dotenv
import platform
import sys


# Sanila needs to set the working directory explicity (macOS) but it's not necessary on Windows or WSL.
# added this macro so we don't have to manually comment/uncommment os.chdir() every time.

IS_MAC = platform.system() == "Darwin"

if IS_MAC:
    os.chdir("/Users/sanilachowdhury/Desktop/ai_agents")
    print("Running on macOS")
else:
    print("Not running on macOS, current directory:", os.getcwd())


load_dotenv()


gemini_key = os.getenv("GEMINI_API_KEY")
skyvern_key = os.getenv("SKYVERN_API_KEY")
phil_key = os.getenv("PHIL-KEY")


os.environ["ENABLE_AUTH"] = "false"

# from skyvern.forge.sdk import SkyvernForge
from skyvern import Skyvern
import nest_asyncio
import asyncio

# Apply nest_asyncio to allow nested event loops in Colab
nest_asyncio.apply()

# 2. Map it to the environment variables Skyvern/LiteLLM looks for
os.environ["GEMINI_API_KEY"] = gemini_key
# os.environ["GOOGLE_API_KEY"] = gemini_key  # Some versions prefer this
# Optional: Force Skyvern to use a specific model
os.environ["LLM_KEY"] = "gemini/gemini-1.5-flash"
print(f"Key check: {'FOUND' if os.environ.get('GEMINI_API_KEY') else 'MISSING'}")
# print("Keys loaded and environment configured.")

### A website we're feeding into our agent to fill in the field (this website is used for testing purpose)

In [ ]:
url = "https://demo.applitools.com/"

### Agent takes in the filepath where the user's resume is stored and we prompt the agent to extract the info in a JSON that easy for both LLM and humans to read

In [ ]:
import asyncio
import nest_asyncio
import os
from google import genai
from IPython.display import display, Markdown

nest_asyncio.apply()
client_genai = genai.Client(api_key=gemini_key)

def parse_resume(file_path, prompt="Please parse this resume and return its contents in a clear, readable format. Use sections with headers (like Contact, Summary, Experience, Education, Skills) and plain text or bullet points — no JSON or code blocks."):
    """
    Uploads a PDF to Gemini and returns the model's analysis.
    Caches the result so Gemini is only called once per resume.
    """
    cache_path = file_path.replace(".pdf", "_cached.json")

    # delete old cache if it exists to force re-parse with new format
    # if os.path.exists(cache_path):
    #     os.remove(cache_path)
    #     print(f"Cleared old cache at {cache_path}.")

    if os.path.exists(cache_path):
        print(f"Using cached resume data from {cache_path}...")
        with open(cache_path, "r") as f:
            return f.read()

    # upload user's resume
    print(f"Uploading {file_path}...")
    uploaded_file = client_genai.files.upload(
        file=file_path,
        config={"display_name": "User_PDF"}
    )

    # feeds LLM the prompt and uploaded resume PDF
    response = client_genai.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=[uploaded_file, prompt]
    )

    # save to cache so we don't call Gemini again
    with open(cache_path, "w") as f:
        f.write(response.text)
    print(f"Resume parsed and cached to {cache_path}")

    return response.text

# call the function
# result = parse_resume("content/jakes-resume.pdf")
# display(Markdown(f"<div style='font-size: 14px'>{result}</div>"))

## Plans before next team report:
* #### Need to figure out how to see the filled in website, not as a screenshot but in another tab.
* #### Follow our workflow where the user can make corrections to the extracted resume info.

## Team Report #2 (03/31) - Created human in the loop for resume info verification, set up ChromaDB, and opened a window of the job website (dummy url) and filled in a few fields.
### (Replaced Skyvern with Browser_Use). This code will not run on Colab, only locally.

<img src="AI_AGENT_WORKFLOW_REPORT_2.png" width="700">

### Developed a human in the loop where agent takes in the extracted info and ask the user if the info is correct. Until the user is satisifed, the agent will keep updating the extracted info based on user's feedback.

In [ ]:
from IPython.display import display, Markdown

def verify_resume_info(extracted_info):
    current_info = extracted_info
    while True:
        print("\n" + "="*10)
        print("EXTRACTED RESUME INFO:")
        print("="*10)
        display(Markdown(f"<div style='font-size: 16px'>{current_info}</div>"))
        print("="*10)

        user_confirm = input("\nIs this information correct? (y/n): ").strip().lower()
        print(f"Is this information correct? (y/n): {user_confirm}")

        if user_confirm == "y":
            print("\nGreat! Moving forward with this information.")
            break
        else:
            correction = input("\nWhat needs to be corrected or added? Please describe: ").strip()
            print(f"What needs to be corrected or added? Please describe: {correction}")

            print("\nUpdating extracted info...")
            response = client_genai.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=[
                    f"""Here is the currently extracted resume information:
{current_info}
The user wants the following correction or addition:
{correction}
Please update the extracted resume JSON accordingly and return the full updated JSON."""
                ]
            )
            current_info = response.text
            print("\nInfo updated! Let's review it again.")
    return current_info


# # call the function to store the extracted info
# result = parse_resume("content/jakes-resume.pdf")

# # run verify_resume_info to verify and correct
# final_info = verify_resume_info(result)

### Setting up ChromaDB which will store the correct parsed info of user's resume

In [ ]:
import chromadb
import os
import hashlib
from dotenv import load_dotenv

# debug - check if values are loaded
print(f"CHROMA_HOST: {os.getenv('CHROMA_HOST')}")
print(f"CHROMA_TENANT: {os.getenv('CHROMA_TENANT')}")
print(f"CHROMA_DATABASE: {os.getenv('CHROMA_DATABASE')}")
print(f"CHROMA_API_KEY: {'FOUND' if os.getenv('CHROMA_API_KEY') else 'MISSING'}")

chroma_client = chromadb.HttpClient(
    ssl=True,
    host=os.getenv("CHROMA_HOST"),
    tenant=os.getenv("CHROMA_TENANT"),
    database=os.getenv("CHROMA_DATABASE"),
    headers={"x-chroma-token": os.getenv("CHROMA_API_KEY")}
)
collection = chroma_client.get_or_create_collection(name="resumes")
print(f"Connected to ChromaDB! Total resumes stored: {collection.count()}")

def store_resume_in_chromadb(resume_text, file_path):
    
    doc_id = hashlib.md5(file_path.encode()).hexdigest()
    collection.upsert(
        documents=[resume_text],
        ids=[doc_id],
        metadatas=[{"file_path": file_path}]
    )
    print(f"Resume stored! ID: {doc_id}")
    return doc_id

# store the verified resume
# doc_id = store_resume_in_chromadb(final_info, "content/jakes-resume.pdf")
print("Resume stored successfully!")

## Plans for spring break:
* ##### Implementing a function where our agent to fill in the fields from the job website using Skyvern (we have started this process but currently experiencing errors)
* ##### Implementing a loop for our agent to handle when it comes to short-answer response
* ##### Have a prompt that will tell the LLM to generate answer based on the context given and the info it extracted from user's resume
* ##### Have a loop where as long as the answer is too vague, keep asking user for more info and implement a function for LLM to do web search (depending on the short answer question)
* ##### Have LLM generate a short answer response and fill out the field with Skyvern
* ##### IF WE FIND TIME -> Have an evalulation where user can give the LLM feedback at the end of the overall performance and store that in ChromaDB to help Phil improve

## Team Report #3 (04/14) - Resolved the EXHAUSTED API key issue by switching to OpenRouter (paid). Implemented most of the workflow and tested the agent.
### Started the evaluation portion of this project. The agent is able to open a window to the provided URL and fill in fields. 
### It successfully extracts the job title and the company name from the website. If the agent successfully fills out the entire application, it uses those fields to update the CSV file. (Works but unreliably because of the drop-down issue).


### See video-demo.mp4 at https://drive.google.com/file/d/1_tLs-5_bf-JU4mLIjEcZ-HX6wFeDz0mI/view?usp=sharing (It gets stuck on a drop down so I ended up interrupting the cell, which closed the browser.)
### (Video too big for git so I put it in Google Drive)


### Currently testing the agent with the job application at https://job-boards.greenhouse.io/attentive/jobs/4209296009

<img src="AI_AGENT_WORKFLOW_REPORT_3.png" width="700">

### A function to write to a CSV file with details of the job application. If a CSV file does not exist, it will create one.

In [ ]:
import csv
import os

def save_application_history(url, company_name, time_applied, job_title):
    # Sanila needs to set the working directory explicity (macOS) but it's not necessary on Windows or WSL.
    # added this macro so we don't have to manually comment/uncommment os.chdir() every time.
    if IS_MAC:
        csv_path = "/Users/sanilachowdhury/Desktop/ai_agents/job-agent/application_history.csv"
    else:
        csv_path = os.path.join(os.getcwd(), "application_history.csv")

    file_exists = os.path.isfile(csv_path)
    with open(csv_path, mode="a", newline="") as csvfile:
        fieldnames = ["url", "company_name", "time_applied", "job_title"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow({"url": url, "company_name": company_name, "time_applied": time_applied, "job_title": job_title})
    print(f"Application history updated: {company_name} - {job_title}")
    print(f"Saved to: {csv_path}")


### Agent ask user for any additional info that's not indicated on their resume but would be helpful in filling in the fields. That info are stored in ChromaDB.
### e.g. Demographic, Work authorization.

In [ ]:
def collect_and_store_additional_info():
    additional_collection = chroma_client.get_or_create_collection(name="user_additional_info")

    # check if info already exists
    existing = additional_collection.get(ids=["user_info"])
    existing_data = {}
    if existing["documents"]:
        # parse existing values
        for line in existing["documents"][0].split("\n"):
            if ": " in line:
                key, val = line.split(": ", 1)
                existing_data[key.strip()] = val.strip()

    print("\nEnter new values or press Enter to keep existing ones:")

    def prompt(label, key, default=""):
        current = existing_data.get(key, default)
        val = input(f"{label} (current: '{current}'): ").strip()
        return val if val else current

    gender = prompt("Gender (e.g. Male, Female, Non-binary, Prefer not to say)", "Gender")
    race = prompt("Race/Ethnicity (e.g. Asian, White, Hispanic, Black, Prefer not to say)", "Race/Ethnicity")
    veteran_status = prompt("Veteran status (e.g. Not a veteran, Veteran, Prefer not to say)", "Veteran Status")
    disability_status = prompt("Disability status (e.g. No disability, Has disability, Prefer not to say)", "Disability Status")
    work_authorization = prompt("Are you authorized to work in the US? (Yes/No)", "Work Authorization")
    sponsorship = prompt("Do you require sponsorship now or in the future? (Yes/No)", "Requires Sponsorship")
    visa_type = prompt("Visa type if applicable (e.g. H1B, F1 OPT, Green Card, Citizen, N/A)", "Visa Type")
    misc_info = prompt("Any other relevant info you'd like to add?", "Misc Info")
    
    info_text = f"""Gender: {gender or 'Prefer not to say'}
Race/Ethnicity: {race or 'Prefer not to say'}
Veteran Status: {veteran_status or 'Prefer not to say'}
Disability Status: {disability_status or 'Prefer not to say'}
Work Authorization: {work_authorization or 'Yes'}
Requires Sponsorship: {sponsorship or 'No'}
Visa Type: {visa_type or 'N/A'}
Misc Info: {misc_info or 'None'}""".strip()

    additional_collection.upsert(
        documents=[info_text],
        ids=["user_info"],
        metadatas=[{"type": "additional_info"}]
    )
    print("✅ Additional info updated in ChromaDB!")
    return info_text

def get_additional_info():
    """Fetch additional user info from ChromaDB."""
    additional_collection = chroma_client.get_or_create_collection(name="user_additional_info")
    results = additional_collection.get(ids=["user_info"])
    if results["documents"]:
        return results["documents"][0]
    return None

### Prints the additional user info stored in ChromaDB (used for testing purpose)

In [ ]:
def print_additional_info():
    additional_collection = chroma_client.get_or_create_collection(name="user_additional_info")
    results = additional_collection.get(ids=["user_info"])
    
    if not results["documents"]:
        print("❌ No additional info found in ChromaDB.")
        return None
    
    info = results["documents"][0]
    print("\n" + "="*40)
    print("📋 YOUR STORED ADDITIONAL INFO:")
    print("="*40)
    print(info)
    print("="*40)
    return info

print_additional_info()

### Agent handling short-answer questions field

In [ ]:
def short_answer_field(field_name: str, resume_text: str, job_url: str) -> str:
    print(f"\n📝 Short answer field detected: '{field_name}'")

    # attempt to generate answer based on resume info
    response = client_genai.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=[
            f"Based on this resume, generate a concise answer for the job application field: '{field_name}'.\n"
            f"Resume:\n{resume_text}\n\n"
            f"If the resume does not have enough info to answer this field, respond with exactly: NEEDS_MORE_INFO\n"
            f"Otherwise respond with only the answer, nothing else."
        ]
    )
    answer = response.text.strip()

    # if there isn't enough info in the resume, ask the user for more info
    if "NEEDS_MORE_INFO" in answer.upper():
        print(f"⚠️ Resume doesn't have enough info for '{field_name}'.")
        user_info = input(f"Please provide info for '{field_name}' (or press Enter to skip): ").strip()

        if not user_info:
            print(f"⏭️ Skipping '{field_name}'")
            return None

        # check if info is too vague
        vague_check = client_genai.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=[
                f"Is this answer too vague to fill in a job application field called '{field_name}'? "
                f"Answer only YES or NO.\nAnswer: {user_info}"
            ]
        )
        while "YES" in vague_check.text.upper():
            print("⚠️ That answer is too vague.")
            user_info = input(f"Please provide a more specific answer for '{field_name}': ").strip()
            if not user_info:
                return None
            vague_check = client_genai.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=[
                    f"Is this answer too vague to fill in a job application field called '{field_name}'? "
                    f"Answer only YES or NO.\nAnswer: {user_info}"
                ]
            )

        # check if company research is needed
        research_check = client_genai.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=[
                f"Does answering the job application field '{field_name}' require researching the company at {job_url}? "
                f"Answer only YES or NO."
            ]
        )

        extra_context = ""
        if "YES" in research_check.text.upper():
            print(f"🔍 Researching company for '{field_name}'...")
            research = client_genai.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=[
                    f"Summarize relevant info about the company at {job_url} "
                    f"that would help answer the job application field: '{field_name}'."
                ]
            )
            extra_context = research.text.strip()
            print(f"✅ Company research done.")

        # generate final answer with user info + extra context
        final_response = client_genai.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=[
                f"Generate a concise job application answer for the field '{field_name}'.\n"
                f"Resume:\n{resume_text}\n"
                f"User provided: {user_info}\n"
                f"Extra context: {extra_context}\n"
                f"Return only the answer, nothing else."
            ]
        )
        answer = final_response.text.strip()

    return answer

### Agent fills in the application field

In [ ]:
import json
import re
from datetime import datetime
from browser_use import Agent, BrowserProfile
from browser_use.browser import BrowserSession
from browser_use.llm import ChatOpenAI


async def fill_application(job_url: str, resume_text: str, additional_info: str = ""):
    llm = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_KEY"),
        model="gemini-2.5-flash-lite",
    )

    browser_session = BrowserSession(
        browser_profile=BrowserProfile(headless=False, keep_alive=True)
    )
    await browser_session.start()

    # open browser to the job URL
    agent_open = Agent(
        task=f"Go to {job_url} and wait. Do not fill anything.",
        llm=llm,
        browser_session=browser_session,
        max_steps=3,
        use_vision=False,
    )
    await agent_open.run()

    # wait for user to sign in
    signed_in = input("Are you signed in (if needed)? (y/n): ").strip().lower()
    while signed_in != "y" and signed_in != "yes":
        signed_in = input("Please sign in and then enter y to continue: ").strip().lower()

    # ask user to upload resume manually
    input("Please upload your resume manually in the browser if required, then press Enter to continue...")

    # detect short answer fields, company name, and job title
    agent_detect = Agent(
        task=(
            f"IMPORTANT - USE THIS INFO FOR DEMOGRAPHIC AND AUTHORIZATION FIELDS. IF YOU ARE UNSURE OF WHAT TO DO, SIMPLE SKIP THE FIELD (mention it to the user) AND DO NOT GO BACK TO THAT SKIPPED FIELD (no matter what):\n"
            f"{additional_info}\n\n"
            f"The page {job_url} is already open. "
            "Look at the page and do the following:\n"
            "1. Find the company name and job title from the page.\n"
            "2. Find any SHORT ANSWER or ESSAY fields (e.g. 'Why do you want to work here?', 'Tell us about yourself', 'Cover letter', etc.).\n"
            "Return ONLY a JSON like this:\n"
            '{"company_name": "Google", "job_title": "Software Engineer", "short_answer_fields": ["field1", "field2"]}'
            "\nIf there are no short answer fields, return: "
            '{"company_name": "Google", "job_title": "Software Engineer", "short_answer_fields": []}'
        ),
        llm=llm,
        browser_session=browser_session,
        use_vision=True,
        max_steps=5,
        max_actions_per_step=3,
    )
    detect_result = await agent_detect.run()
    detect_final = detect_result.final_result()

    # handle short answer fields and extract company/job title
    short_answer_context = ""
    detected_company = "Unknown"
    detected_job_title = "Unknown"
    try:
        json_match = re.search(r'\{.*\}', detect_final, re.DOTALL)
        detected = json.loads(json_match.group()) if json_match else json.loads(detect_final)
        short_answer_fields = detected.get("short_answer_fields", [])
        detected_company = detected.get("company_name", "Unknown")
        detected_job_title = detected.get("job_title", "Unknown")
        print(f"🏢 Company: {detected_company} | 💼 Job Title: {detected_job_title}")

        if short_answer_fields:
            print(f"\n📋 Short answer fields found: {short_answer_fields}")
            answers = {}
            for field in short_answer_fields:
                answer = short_answer_field(field, resume_text, job_url)
                if answer:
                    answers[field] = answer
            if answers:
                short_answer_context = "\n".join([f"{k}: {v}" for k, v in answers.items()])
    except Exception as e:
        print(f"⚠️ Could not detect page info: {e}")

    # fill all fields
    agent_fill = Agent(
        task=(
            f"IMPORTANT - USE THIS INFO FOR DEMOGRAPHIC AND AUTHORIZATION FIELDS:\n"
            f"{additional_info}\n\n"
            f"The page {job_url} is already open. "
            f"Use this resume to fill out the application:\n{resume_text}\n\n"
            + (f"Use these pre-generated answers for short answer fields:\n{short_answer_context}\n\n" if short_answer_context else "")
            +
            "Instructions:\n"
            "1. Find every empty field on the page.\n"
            "2. If the field can be answered from the resume, fill it in.\n"
            "3. For DROPDOWN fields:\n"
            "   - Click the dropdown to open it.\n"
            "   - Immediately click on one of the visible options — do NOT scroll, do NOT search, do NOT re-click the dropdown.\n"
            "   - After clicking an option, move on to the next field immediately.\n"
            "   - If after 2 attempts the dropdown still does not respond, use JavaScript to select the value directly and move on.\n"
            "   - NEVER attempt the same dropdown more than 2 times.\n"
            "4. For SENSITIVE fields (Gender, Race, Veteran status, Disability status, Ethnicity):\n"
            "   - Use the additional user info provided above to fill these fields.\n"
            "   - If not found in additional info, pick the FIRST available option.\n"
            "5. For SPONSORSHIP/WORK AUTHORIZATION fields:\n"
            "   - Use the work authorization and sponsorship info from the additional user info above.\n"
            "   - If unclear, select 'Yes' for authorization and 'No' for sponsorship as default.\n"
            "6. For COUNTRY fields: select 'United States' unless the resume says otherwise.\n"
            "7. For FILE UPLOAD fields (resume, CV): skip them — the user has already uploaded manually.\n"
            "8. For other FILE UPLOAD fields (cover letter, portfolio, other documents): skip them.\n"
            "9. For SHORT ANSWER or ESSAY fields: use the pre-generated answers provided above.\n"
            "10. DO NOT click 'Submit', 'Submit application', 'Apply', or ANY button that submits the form. This is strictly forbidden.\n"
            "11. When all fields are processed, return ONLY a JSON like this:\n"
            '{"filled": ["field1", "field2"], "empty": ["field3"], "skipped_uploads": ["resume", "cover letter"], "company_name": "Google", "job_title": "Software Engineer"}'
        ),
        llm=llm,
        browser_session=browser_session,
        use_vision=True,
        max_steps=50,
        max_actions_per_step=10,
        include_attributes=["id", "name", "type", "placeholder", "aria-label", "label", "role", "data-testid"],
    )

    result = await agent_fill.run()
    final = result.final_result()
    print("Agent result:", final)

    # save the applied job to CSV if all fields are filled
    try:
        json_match = re.search(r'\{.*\}', final, re.DOTALL)
        parsed = json.loads(json_match.group()) if json_match else json.loads(final)

        empty_fields = parsed.get("empty", [])

        # Use detected values as source of truth, fall back to agent's JSON if unknown
        company_name = detected_company if detected_company != "Unknown" else parsed.get("company_name", "Unknown")
        job_title = detected_job_title if detected_job_title != "Unknown" else parsed.get("job_title", "Unknown")

        if empty_fields:
            print(f"⚠️ Empty fields remaining: {empty_fields}")
        else:
            print("✅ All fields filled!")
            save_application_history(
                url=job_url,
                company_name=company_name,
                time_applied=datetime.today().strftime("%Y-%m-%d %H:%M:%S"),
                job_title=job_title,
            )
    except Exception as e:
        print(f"⚠️ Could not parse result: {e}")
        save_application_history(
            url=job_url,
            company_name=detected_company,
            time_applied=datetime.today().strftime("%Y-%m-%d %H:%M:%S"),
            job_title=detected_job_title,
        )

    return final

### Main function where user uses the agent to help apply to jobs

In [ ]:
from IPython.display import display, Markdown
async def apply_to_job_with_agent():
    # first check if user wants to upload a new resume or use existing one in ChromaDB
    resume_exists = input("Would you like to upload a new resume? Enter y if this is your first time (y/n): ")
    while resume_exists != "y" and resume_exists != "n":
        resume_exists = input("Invalid input. Please enter y or n: ")

    if resume_exists == "y":
        file_path = input("Please enter the path to your resume PDF: ").strip()
        extracted_info = parse_resume(file_path)
        print("\n📄 Resume extracted info:")
        print(extracted_info)
        
        verified_info = verify_resume_info(extracted_info)
        store_resume_in_chromadb(verified_info, file_path)
        resume_text = verified_info
    else:
        print("Using existing resume data. Moving on to job application...")
        results = collection.get(limit=1, include=["documents"])
        if not results["documents"]:
            raise RuntimeError("No resume found in ChromaDB. Please upload a resume first.")
        resume_text = results["documents"][0]
        print("✅ Resume fetched from ChromaDB")

    # collect or fetch additional info
    additional_info = get_additional_info()
    if not additional_info:
        print("\n📋 No additional info found. Let's collect it now.")
        additional_info = collect_and_store_additional_info()
    else:
        print_additional_info()
        update = input("\nWould you like to update your demographic/work authorization info? (y/n): ").strip().lower()
        if update == "y":
            additional_info = collect_and_store_additional_info()

    url = input("Please enter the URL of the job application you are applying to: ").strip()
    await fill_application(url, resume_text, additional_info) # call this function to run the whole agent process

await apply_to_job_with_agent()

Using cached resume data from content/jakes-resume_cached.json...

📄 Resume extracted info:
Here's the parsed resume content:

**Contact**

*   **Name:** Jake Ryan
*   **Phone:** 123-456-7890
*   **Email:** jake@su.edu
*   **LinkedIn:** linkedin.com/in/jake
*   **GitHub:** github.com/jake

**Education**

*   **Southwestern University**
    *   Bachelor of Arts in Computer Science, Minor in Business
    *   Georgetown, TX
    *   Aug. 2018 - May 2021
*   **Blinn College**
    *   Associate's in Liberal Arts
    *   Bryan, TX
    *   Aug. 2014 - May 2018

**Experience**

*   **Undergraduate Research Assistant**
    *   Texas A&M University
    *   College Station, TX
    *   June 2020 - Present
    *   Developed a REST API using FastAPI and PostgreSQL to store data from learning management systems
    *   Developed a full-stack web application using Flask, React, PostgreSQL and Docker to analyze GitHub data
    *   Explored ways to visualize GitHub collaboration in a classroom setting

*

<div style='font-size: 16px'>Here's the parsed resume content:

**Contact**

*   **Name:** Jake Ryan
*   **Phone:** 123-456-7890
*   **Email:** jake@su.edu
*   **LinkedIn:** linkedin.com/in/jake
*   **GitHub:** github.com/jake

**Education**

*   **Southwestern University**
    *   Bachelor of Arts in Computer Science, Minor in Business
    *   Georgetown, TX
    *   Aug. 2018 - May 2021
*   **Blinn College**
    *   Associate's in Liberal Arts
    *   Bryan, TX
    *   Aug. 2014 - May 2018

**Experience**

*   **Undergraduate Research Assistant**
    *   Texas A&M University
    *   College Station, TX
    *   June 2020 - Present
    *   Developed a REST API using FastAPI and PostgreSQL to store data from learning management systems
    *   Developed a full-stack web application using Flask, React, PostgreSQL and Docker to analyze GitHub data
    *   Explored ways to visualize GitHub collaboration in a classroom setting

*   **Information Technology Support Specialist**
    *   Southwestern University
    *   Georgetown, TX
    *   Sep. 2018 - Present
    *   Communicate with managers to set up campus computers used on campus
    *   Assess and troubleshoot computer problems brought by students, faculty and staff
    *   Maintain upkeep of computers, classroom equipment, and 200 printers across campus

*   **Artificial Intelligence Research Assistant**
    *   Southwestern University
    *   Georgetown, TX
    *   May 2019 - July 2019
    *   Explored methods to generate video game dungeons based off of *The Legend of Zelda*
    *   Developed a game in Java to test the generated dungeons
    *   Contributed 50K+ lines of code to an established codebase via Git
    *   Conducted a human subject study to determine which video game dungeon generation technique is enjoyable
    *   Wrote an 8-page paper and gave multiple presentations on-campus
    *   Presented virtually to the World Conference on Computational Intelligence

**Projects**

*   **Gitlytics**
    *   Python, Flask, React, PostgreSQL, Docker
    *   June 2020 - Present
    *   Developed a full-stack web application using with Flask serving a REST API with React as the frontend
    *   Implemented GitHub OAuth to get data from user's repositories
    *   Visualized GitHub data to show collaboration
    *   Used Celery and Redis for asynchronous tasks

*   **Simple Paintball**
    *   Spigot API, Java, Maven, TravisCI, Git
    *   May 2018 - May 2020
    *   Developed a Minecraft server plugin to entertain kids during free time for a previous job
    *   Implemented continuous delivery using TravisCI to build the plugin upon new a release
    *   Published plugin to websites gaining 2K+ downloads and an average 4.5/5-star review
    *   Collaborated with Minecraft server administrators to suggest features and get feedback about the plugin

**Technical Skills**

*   **Languages:** Java, Python, C/C++, SQL (Postgres), JavaScript, HTML/CSS, R
*   **Frameworks:** React, Node.js, Flask, JUnit, WordPress, Material-UI, FastAPI
*   **Developer Tools:** Git, Docker, TravisCI, Google Cloud Platform, VS Code, Visual Studio, PyCharm, IntelliJ, Eclipse
*   **Libraries:** pandas, NumPy, Matplotlib</div>

Is this information correct? (y/n): n
What needs to be corrected or added? Please describe: my first name is Blake and my phone number is 718-999-0000

Updating extracted info...

Info updated! Let's review it again.

EXTRACTED RESUME INFO:


<div style='font-size: 16px'>```json
{
  "contact": {
    "name": "Blake Ryan",
    "phone": "718-999-0000",
    "email": "jake@su.edu",
    "linkedin": "linkedin.com/in/jake",
    "github": "github.com/jake"
  },
  "education": [
    {
      "institution": "Southwestern University",
      "degree": "Bachelor of Arts in Computer Science, Minor in Business",
      "location": "Georgetown, TX",
      "dates": "Aug. 2018 - May 2021"
    },
    {
      "institution": "Blinn College",
      "degree": "Associate's in Liberal Arts",
      "location": "Bryan, TX",
      "dates": "Aug. 2014 - May 2018"
    }
  ],
  "experience": [
    {
      "title": "Undergraduate Research Assistant",
      "company": "Texas A&M University",
      "location": "College Station, TX",
      "dates": "June 2020 - Present",
      "description": [
        "Developed a REST API using FastAPI and PostgreSQL to store data from learning management systems",
        "Developed a full-stack web application using Flask, React, PostgreSQL and Docker to analyze GitHub data",
        "Explored ways to visualize GitHub collaboration in a classroom setting"
      ]
    },
    {
      "title": "Information Technology Support Specialist",
      "company": "Southwestern University",
      "location": "Georgetown, TX",
      "dates": "Sep. 2018 - Present",
      "description": [
        "Communicate with managers to set up campus computers used on campus",
        "Assess and troubleshoot computer problems brought by students, faculty and staff",
        "Maintain upkeep of computers, classroom equipment, and 200 printers across campus"
      ]
    },
    {
      "title": "Artificial Intelligence Research Assistant",
      "company": "Southwestern University",
      "location": "Georgetown, TX",
      "dates": "May 2019 - July 2019",
      "description": [
        "Explored methods to generate video game dungeons based off of *The Legend of Zelda*",
        "Developed a game in Java to test the generated dungeons",
        "Contributed 50K+ lines of code to an established codebase via Git",
        "Conducted a human subject study to determine which video game dungeon generation technique is enjoyable",
        "Wrote an 8-page paper and gave multiple presentations on-campus",
        "Presented virtually to the World Conference on Computational Intelligence"
      ]
    }
  ],
  "projects": [
    {
      "name": "Gitlytics",
      "technologies": "Python, Flask, React, PostgreSQL, Docker",
      "dates": "June 2020 - Present",
      "description": [
        "Developed a full-stack web application using with Flask serving a REST API with React as the frontend",
        "Implemented GitHub OAuth to get data from user's repositories",
        "Visualized GitHub data to show collaboration",
        "Used Celery and Redis for asynchronous tasks"
      ]
    },
    {
      "name": "Simple Paintball",
      "technologies": "Spigot API, Java, Maven, TravisCI, Git",
      "dates": "May 2018 - May 2020",
      "description": [
        "Developed a Minecraft server plugin to entertain kids during free time for a previous job",
        "Implemented continuous delivery using TravisCI to build the plugin upon new a release",
        "Published plugin to websites gaining 2K+ downloads and an average 4.5/5-star review",
        "Collaborated with Minecraft server administrators to suggest features and get feedback about the plugin"
      ]
    }
  ],
  "technical_skills": {
    "languages": [
      "Java",
      "Python",
      "C/C++",
      "SQL (Postgres)",
      "JavaScript",
      "HTML/CSS",
      "R"
    ],
    "frameworks": [
      "React",
      "Node.js",
      "Flask",
      "JUnit",
      "WordPress",
      "Material-UI",
      "FastAPI"
    ],
    "developer_tools": [
      "Git",
      "Docker",
      "TravisCI",
      "Google Cloud Platform",
      "VS Code",
      "Visual Studio",
      "PyCharm",
      "IntelliJ",
      "Eclipse"
    ],
    "libraries": [
      "pandas",
      "NumPy",
      "Matplotlib"
    ]
  }
}
```</div>

Is this information correct? (y/n): y

Great! Moving forward with this information.
Resume stored! ID: b7a0358df030937c049bb9754117f552
🏢 Company: Attentive | 💼 Job Title: AI Intern


2026-04-13T14:07:55.409103Z [warning  ]  ⚠️ LLM error (ModelProviderError: 1 validation error for AgentOutput
  Invalid JSON: EOF while parsing an object at line 6 column 24 [type=json_invalid, input_value='{\n  "thinking": "The pr... "current_plan_item": 1', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid) but no fallback_llm configured [browser_use.Agent🅰 d064 ⇢ 🅑 ff66 🅣 2D]
2026-04-13T14:07:55.410005Z [warning  ]  ❌ Result failed 1/6 times: 1 validation error for AgentOutput
  Invalid JSON: EOF while parsing an object at line 6 column 24 [type=json_invalid, input_value='{\n  "thinking": "The pr... "current_plan_item": 1', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid [browser_use.Agent🅰 d064 ⇢ 🅑 ff66 🅣 2D]
2026-04-13T14:08:51.373481Z [warning  ]  ⚠️ Element index 2778 not available - page may have changed. Try refreshing browser state. [browser_use.tools.service]


Agent result: I have filled out the following fields: First Name, Last Name, Preferred First Name, Email, Phone, Country, LinkedIn Profile, Website, Sponsorship, Gender, Race, Veteran status, GDPR consent, Disability Status, and Misc Info. The 'Visa Type' field could not be found or interacted with after multiple attempts. I have not clicked the submit button.
⚠️ Could not parse result: Expecting value: line 1 column 1 (char 0)
Application history updated: Attentive - AI Intern
Saved to: /home/rb4271/ai-agents/job-agent/application_history.csv


In [ ]:
#function takes in a list of single page urls and multi page urls.
#IMPORTANT: each resume in test_resumes/ folder should following this naming convention: dummy_resume_1.pdf  (only change the number)
async def test_agent_withdummy_resumes(n_resumes, single_page_urls, multi_page_urls):
    test_resumes_dir = "content/test_resumes/"
    for i in range(n_resumes):
        print(f"\n=== TESTING RESUME {i+1}/{n_resumes} ===")
        resume_path = f"{test_resumes_dir}dummy_resume_{i+1}.pdf"
        resume_text = parse_resume(resume_path)

        print("\nTesting on single-page application...")
        await fill_application(single_page_urls[i % len(single_page_urls)], resume_text)

        print("\nTesting on multi-page application...")
        await fill_application(multi_page_urls[i % len(multi_page_urls)], resume_text)

    print(f"Done testing {n_resumes} resumes on single and multi-page applications.")

## Plans before next team report:
* #### Continue working on evaluations:
    * ##### So far, we evaluated the component level of our agent where it parsed the resume correctly, leveraging LLM calls when needed (for additional info from the user and short-answer fields), and appending the filled-out job position to a CSV file.
    * ##### At the end-to-end level, we tested how our agent handles different types of fields (right now it's unreliable on drop-down field because it sometimes get stuck and keeps choosing random or incorrect options. It even keeps going back to the same field after filling it out so we would need to change our prompt), preventing the agent from pressing submit once all fields are filled, and how accurately it fills in fields with the correct information retrieved from the resume and the user's additional input.
    * ##### Next, we need to support multi-page job sites and navigation across different job boards (Greenhouse vs. Workday), as the agent currently only works on single-page applications.


* #### Short-answer field that involves HITL:
    * ##### We still need to integrate HITL, especially when the Agent lacks info for a field. We have already implemented basic short-answer handling, but need to extend it to asking the user for more info and triggering a web search when the question requires external or company-specific knowledge (e.g., "Why do you want to work at X?"), incorporating retrieved results into the generated response.

## Team Report #4 (04/21) - Evals